In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 25. text — Modeltext text text [CPU/GPU Benchmark ⑪]

> **Learning Goals**
> - FP32, FP16, BF16, INT8, INT4text Precision·text·Speed text text
> - text text $q = \mathrm{round}(x/s) + z$text textdegreestext
> - GPTQ, AWQ, GGUF text text text text

## 25.1 text text

LLMtext text·text text:
- LLaMA-7B FP16: 14GB
- Inference text text KV Cache
- text/text text text

text text:
- FP16 → INT8: text text, Speed 2text
- FP16 → INT4: text 1/4, Speed 3-4text

## 25.2 text text

| text | text | text | text | text |
|---|---|---|---|---|
| FP32 | 1 | 8 | 23 | 32 bit |
| FP16 | 1 | 5 | 10 | 16 bit |
| BF16 | 1 | 8 | 7 | 16 bit |
| FP8 (E4M3) | 1 | 4 | 3 | 8 bit |

- FP16: Precision text text text (text text)
- BF16: FP32text text text, Precision text → LLM Trainingtext text


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# text text Comparison
print("textPoint text text:")
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16), ('BF16', torch.bfloat16)]:
    info = torch.finfo(dtype)
    print(f"  {name}: min={info.min:.2e}, max={info.max:.2e}, eps={info.eps:.2e}")

# FP16 text text
print("\nFP16 text:")
print(f"  2^15 = {float(torch.tensor(2**15, dtype=torch.float16))}")
print(f"  2^16 = {float(torch.tensor(2**16, dtype=torch.float16))}  # inf!")
print(f"  BF16 2^16 = {float(torch.tensor(2**16, dtype=torch.bfloat16))}  # text")


## 25.3 text text text

FP32 Valuetext INT8 (0~255 text -128~127)text text:

**text text** (zero-point = 0):
$$q = \mathrm{round}(x / s), \quad s = \frac{\max|x|}{127}$$
$$\hat{x} = s \cdot q$$

**text text** (zero-point text):
$$q = \mathrm{round}(x / s) + z$$
$$s = \frac{\max(x) - \min(x)}{255}, \quad z = -\mathrm{round}(\min(x) / s)$$

text Error: $\epsilon = x - \hat{x}$.


In [ ]:
# text text
def quantize_symmetric(x, n_bits=8):
    """Symmetric Quantization."""
    qmax = 2 ** (n_bits - 1) - 1  # 127 for INT8
    scale = x.abs().max() / qmax
    q = torch.round(x / scale).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.to(torch.float32) * scale

# Test
torch.manual_seed(0)
x = torch.randn(100) * 5  # Mean 0, Standard Deviation 5
q, scale = quantize_symmetric(x, n_bits=8)
x_hat = dequantize_symmetric(q, scale)
error = (x - x_hat).abs()
print(f"text text: [{x.min():.3f}, {x.max():.3f}]")
print(f"Quantization text: [{q.min()}, {q.max()}]")
print(f"text: {scale:.6f}")
print(f"Mean Absolute error: {error.mean():.6f}")
print(f"Maximum Absolute error: {error.max():.6f}")
print(f"text Error: {(error.mean() / x.abs().mean() * 100):.3f}%")

# text text text Comparison
print("\ntext text Quantization Error:")
for n_bits in [8, 4, 2]:
    q, scale = quantize_symmetric(x, n_bits=n_bits)
    x_hat = dequantize_symmetric(q, scale)
    err = (x - x_hat).abs().mean()
    print(f"  INT{n_bits}: Mean Error = {err:.6f} ({err/x.abs().mean()*100:.2f}%)")


## 25.4 Per-tensor, Per-channel, Per-group

- **Per-tensor**: text text text scale. text textdegrees text.
- **Per-channel**: text(text/text)text scale. textdegrees text, text text text.
- **Per-group**: $g$text text scale. text. LLM text text.

text: 4-bit text group_size=128 → 128text Valuetext scale (FP16) text.


In [ ]:
# Per-group text
def quantize_per_group(x, n_bits=4, group_size=128):
    """text Quantization."""
    qmax = 2 ** (n_bits - 1) - 1
    # text text
    orig_shape = x.shape
    x_flat = x.reshape(-1, group_size)
    scales = x_flat.abs().max(dim=1, keepdim=True).values / qmax
    q = torch.round(x_flat / scales).clamp(-qmax - 1, qmax)
    return q, scales, orig_shape

def dequantize_per_group(q, scales, orig_shape):
    return (q * scales).reshape(orig_shape)

# Comparison: per-tensor vs per-group
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.1  # Weight Matrix

# per-tensor (4-bit)
qmax = 7  # 4-bit symmetric
scale_t = W.abs().max() / qmax
q_t = torch.round(W / scale_t).clamp(-8, 7)
W_hat_t = q_t * scale_t
err_t = (W - W_hat_t).abs().mean()

# per-group (4-bit, group=128)
q_g, scales_g, shape = quantize_per_group(W, n_bits=4, group_size=128)
W_hat_g = dequantize_per_group(q_g, scales_g, shape)
err_g = (W - W_hat_g).abs().mean()

print(f"4-bit text Error:")
print(f"  Per-tensor: {err_t:.6f} ({err_t/W.abs().mean()*100:.2f}%)")
print(f"  Per-group (128): {err_g:.6f} ({err_g/W.abs().mean()*100:.2f}%)")
print("\n=> Per-grouptext text text. LLM Quantization Standard.")


## 25.5 PTQ (Post-Training Quantization) vs QAT

**PTQ**: Training text text. text textdegrees text text.

**QAT (Quantization-Aware Training)**: text text Training. text text text.

LLMtext text PTQ text (Training text text text).

## 25.6 GPTQ, AWQ

### GPTQ
- 2text Error text text
- text text (text text)
- Hessian Matrix text Error text
- INT4textdegrees PPL text text text

### AWQ (Activation-aware Weight Quantization)
- text text(text text)text FP16 text
- textValue text textdegrees text
- GPTQtext text text


## 25.7 QLoRA — 4-bit text + LoRA

QLoRA (Dettmers et al., 2023):
1. text Modeltext 4-bit NF4text text (text)
2. LoRA text FP16/BF16text Training
3. text: 7B Modeltext text 24GB GPUtext text text

NF4 (NormalFloat 4-bit): text textDistributiontext text text text 4-bit text.


In [ ]:
# QLoRA text text
def qlora_memory(base_params_b, adapter_ratio=0.01):
    """QLoRA Memory Estimation (GB).
    base_params_b: text Parameter Count (text: 10text)
    adapter_ratio: text text Ratio
    """
    # text: 4-bit = 0.5 bytes
    base_mem = base_params_b * 0.5
    # text: FP16 + AdamW (4text) = 8 bytes
    adapter_mem = base_params_b * adapter_ratio * 8
    # textValue (text)
    activation = 2  # GB
    return base_mem + adapter_mem + activation

# Comparison: text text vs QLoRA
print(f"{'Model':>10} | {'text FT FP16 (GB)':>15} | {'QLoRA 4-bit (GB)':>17}")
print("-" * 50)
for name, n_b in [('7B', 7), ('13B', 13), ('30B', 30), ('70B', 70)]:
    full = n_b * 2 * 4 + 4  # FP16 + AdamW FP32 (4text) + text
    qlora = qlora_memory(n_b, 0.01)
    print(f"{name:>10} | {full:>15.1f} | {qlora:>17.1f}")
print("\n=> QLoRAtext 70B Modeldegrees text 48GB GPUtext text text.")


## 25.8 [CPU/GPU Benchmark ⑪] Precisiontext Inference


In [ ]:
# Precisiontext Inference Speed Comparison
from llm_math.bench import time_fn

# text text Inference
def bench_linear(d_in, d_out, batch_size, dtype, device='cpu'):
    """LinearLayer Inference Time."""
    model = nn.Linear(d_in, d_out).to(dtype=dtype, device=device)
    x = torch.randn(batch_size, d_in, dtype=dtype, device=device)
    def run():
        with torch.no_grad():
            return model(x)
    return run

print(f"{'dtype':>10} | {'CPU (ms)':>10} | {'Memory (MB)':>12}")
print("-" * 40)
d_in, d_out, bs = 4096, 4096, 8
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16)]:
    if dtype == torch.float16 and not torch.cuda.is_available():
        # CPUtext FP16text text
        continue
    try:
        run = bench_linear(d_in, d_out, bs, dtype, 'cpu')
        # warmup
        run()
        res = time_fn(run, device='cpu', warmup=2, repeat=5)
        # Memory (text)
        mem = d_in * d_out * (4 if dtype == torch.float32 else 2) / 1024**2
        print(f"{name:>10} | {res['mean_ms']:>10.3f} | {mem:>12.2f}")
    except Exception as e:
        print(f"{name:>10} | {'N/A':>10} | error: {e}")

print("\n=> FP16text Memory text, CPUtext Speed Improvement text (GPUtext text Improvement).")


## 25.9 Key Takeaways

| text | text | text text | textdegrees text |
|---|---|---|---|
| FP16 | 16 | 2x | text text |
| BF16 | 16 | 2x | text text |
| INT8 | 8 | 4x | text |
| INT4 (GPTQ/AWQ) | 4 | 8x | text |
| QLoRA (NF4) | 4 | 8x | text |

## Exercises

1. INT8 text text text Outputtext MSEtext text.
2. Per-tensor vs Per-channel vs Per-group text Errortext Comparisontext.
3. text text 32, 64, 128, 256text text text Errortext Comparisontext.
4. FP32 vs FP16 Inference Speedtext CPUtext GPUtext text Comparisontext.
5. QLoRAtext text text Comparisontext text text text 7B, 13B, 70B Modeltext text Calculationtext.

> Solutions: `solutions/ch25_solutions.ipynb`
